# Module 1: Data Loading and Augmentation Using Keras

> **Assignment (8 points):** Build a Keras data pipeline with a custom generator and augmentation.

In [ ]:
import os, sys, pathlib, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
PROJECT_ROOT = pathlib.Path(os.path.abspath(os.getcwd()))
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / 'images_dataSAT').is_dir():
        PROJECT_ROOT = p
        break
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
import os, random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf

base_directory = './images_dataSAT'
random.seed(42)

### Task 1: Create `all_image_paths` — paths of all files in both folders (`class_0_non_agri` and `class_1_agri`) inside the base directory

In [ ]:
all_image_paths = []
for folder_name in ['class_0_non_agri', 'class_1_agri']:
    folder_path = os.path.join(base_directory, folder_name)
    all_image_paths.extend([os.path.join(folder_path, f) for f in os.listdir(folder_path)
                            if os.path.isfile(os.path.join(folder_path, f))])

print('Total image paths:', len(all_image_paths))
print(all_image_paths[:3])

### Task 2: Bind paths & labels with `zip` into `temp`, then randomly select and print 5 image paths with their labels

In [ ]:
labels = [0 if 'class_0' in p else 1 for p in all_image_paths]

temp = list(zip(all_image_paths, labels))

selected = random.sample(temp, 5)
for path, label in selected:
    print(path, '->', label)

## Custom data generator (with optional on-the-fly augmentation)

In [ ]:
def custom_data_generator(list_of_files, labels, batch_size=8, image_size=(64, 64),
                          shuffle=True, augment=False):
    indices = list(range(len(list_of_files)))
    while True:
        if shuffle:
            random.shuffle(indices)
        for start in range(0, len(indices), batch_size):
            batch_idx = indices[start:start + batch_size]
            batch_images = []
            for i in batch_idx:
                img = np.array(Image.open(list_of_files[i]).convert('RGB').resize(image_size))
                if augment:
                    img_t = tf.image.random_flip_left_right(tf.convert_to_tensor(img))
                    img_t = tf.image.random_flip_up_down(img_t)
                    img_t = tf.image.random_brightness(img_t, max_delta=0.2)
                    img = img_t.numpy()
                batch_images.append(img)
            X_batch = np.stack(batch_images).astype('float32') / 255.0
            y_batch = np.array([labels[i] for i in batch_idx])
            yield X_batch, y_batch

### Task 3: Generate a batch of data (batch size = 8) by invoking `custom_data_generator`

In [ ]:
train_gen = custom_data_generator(all_image_paths, labels, batch_size=8, augment=False)
X_batch, y_batch = next(train_gen)
print('Batch shape:', X_batch.shape)
print('Labels     :', y_batch)

## Augmentation demo — original vs. augmented versions of the same image

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
img = np.array(Image.open(all_image_paths[0]).convert('RGB').resize((64, 64)))
axes[0].imshow(img)
axes[0].set_title('Original')
axes[0].axis('off')
for k in range(3):
    img_t = tf.image.random_flip_left_right(tf.convert_to_tensor(img))
    img_t = tf.image.random_flip_up_down(img_t)
    img_t = tf.image.random_brightness(img_t, max_delta=0.2)
    axes[k + 1].imshow(img_t.numpy())
    axes[k + 1].set_title(f'Augmented {k + 1}')
    axes[k + 1].axis('off')
plt.tight_layout()
plt.show()

### Task 4: Create validation data using a batch size of 8

In [ ]:
split = int(len(all_image_paths) * 0.8)
train_paths = all_image_paths[:split]
val_paths   = all_image_paths[split:]
train_labels = labels[:split]
val_labels   = labels[split:]

val_gen = custom_data_generator(val_paths, val_labels, batch_size=8)
X_val, y_val = next(val_gen)
print('Validation batch shape:', X_val.shape)
print('Validation labels     :', y_val)

## Keras native pipeline (extra): `image_dataset_from_directory`

In [ ]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    base_directory, labels='inferred', label_mode='binary', color_mode='rgb',
    batch_size=8, image_size=(64, 64), validation_split=0.2, subset='training',
    seed=42, shuffle=True)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    base_directory, labels='inferred', label_mode='binary', color_mode='rgb',
    batch_size=8, image_size=(64, 64), validation_split=0.2, subset='validation',
    seed=42, shuffle=True)

for x, y in train_dataset.take(1):
    print('tf.data batch:', x.shape, y.shape)